# GARCH: volatility after non-trading days

Does the market's volatility change after a weekend or a holiday, and does a shock on such a day last as long as one on an ordinary day? Bollerslev and Ghysels (1996) answer with their daily Deutsche mark / British pound returns: a constant variance, GARCH(1,1), GARCH(1,1) with a non-trading-day dummy, and their new periodic GARCH, in which the dummy also changes how shocks feed into tomorrow's variance. Every estimate and statistic of their Table 2 (p. 146) is printed beside Julia's.

The data is `bg96`, from PollisDatasets: the 1,974 daily returns of January 1984 to December 1991, with the paper's non-trading-day dummy. The code is written for this workflow: the plain GARCH(1,1) is fitted with ARCHModels, and the models with the dummy, which no Julia package has, are written out in a few lines, the periodic GARCH among them.

In [ ]:
Pollis.packages("ARCHModels", "Optim", "ForwardDiff", "PollisDatasets", "Plots")

## Set-up

The returns `y`, the dummy `s` (1 on Mondays and other days after no trading, Bollerslev and Ghysels 1996, p. 145), and `replicate`, which prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using ARCHModels, Optim, ForwardDiff, PollisDatasets, Plots, Printf, Statistics, LinearAlgebra

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

bg = dataset("bg96");  # one row per trading day
hasproperty(bg, :nontrading) || error("This workflow needs the nontrading column of bg96: update PollisDatasets")
y = Float64.(bg.return);  # daily percentage returns, 100 [ln(P_t) - ln(P_t-1)]
s = Float64.(bg.nontrading);  # 1 after a weekend or holiday, 0 otherwise
T = length(y);

## 1. The data

Bollerslev and Ghysels (1996, p. 145): 1,974 returns, 456 of them after no trading, roughly 23%, and a first-order autocorrelation of only .009, so the returns are close to unpredictable in the mean. Their variance is another matter: it is larger after non-trading days, and it comes in clusters.

In [ ]:
r1 = sum((y[2:end] .- mean(y)) .* (y[1:end-1] .- mean(y))) / sum(abs2, y .- mean(y));  # first-order autocorrelation
replicate("The data (Bollerslev and Ghysels 1996, p. 145)", [
	("observations", T, "1,974"), ("non-trading days", sum(s), "456"), ("share, %", 100 * mean(s), "23"), ("autocorrelation, lag 1", r1, ".009")])
@printf("\nvariance of the returns: %.3f on ordinary days, %.3f after non-trading days\n", var(y[s .== 0]), var(y[s .== 1]))

plot(y, lw=0.5, color=:gray, label="return", xlabel="trading day (January 1984 to December 1991)", ylabel="%")
scatter!(findall(==(1), s), y[s .== 1], ms=1.5, msw=0, color=:red, label="after no trading")

## 2. The models

All six columns of Table 2 are special cases of one model (Bollerslev and Ghysels 1996, p. 146, equation 14 and the note to Table 2): the mean is μ + μ₁ s(t), and the conditional variance is

σ²(t) = ω + ω₁ s(t) + (α + α₁ s(t)) (ε²(t−1) − ω − ω₁ s(t−1)) + β (σ²(t−1) − ω − ω₁ s(t−1))

so ω + ω₁ s(t) is the level the variance returns to, ω₁ the extra variance after no trading, and α₁ how much less, or more, a shock on such a day feeds into the next day's variance: this last one is the periodic GARCH. Each column sets some parameters to zero.

`qmle` fits a column by quasi-maximum likelihood, the Gaussian likelihood with standard errors robust to fat tails (the sandwich of the Hessian and the outer product of the scores, which the paper reports), and computes the statistics in the lower half of the table. The recursion starts from the sample variance of the returns, as ARCHModels does.

In [ ]:
parameters = ["mu", "mu1", "omega", "omega1", "alpha", "alpha1", "beta"];
statistics = (:AIC, :SIC, :b3, :b4, :Q, :Q2, :MSE, :HMSE, :LL);  # the rows below the estimates in Table 2

"""
	variances(θ)

Residuals and conditional variances of the model above, for θ = (μ, μ₁, ω, ω₁, α, α₁, β).
"""
function variances(θ)
	μ, μ1, ω, ω1, α, α1, β = θ
	ε = y .- μ .- μ1 .* s
	h = similar(ε)
	h[1] = var(y)  # start from the sample variance
	for t in 2:T
		level = ω + ω1 * s[t-1]  # the variance level of day t-1
		h[t] = ω + ω1 * s[t] + (α + α1 * s[t]) * (ε[t-1]^2 - level) + β * (h[t-1] - level)
	end
	ε, h
end

"""
	loglik(θ)

Gaussian log likelihood of each day, or -Inf everywhere if a variance is not positive.
"""
function loglik(θ)
	ε, h = variances(θ)
	all(>(0), h) || return fill(oftype(first(h), -Inf), T)
	-(log(2pi) .+ log.(h) .+ ε .^ 2 ./ h) ./ 2
end

"""
	diagnostics(ε, h, LL, k)

The statistics below the estimates in Table 2: information criteria (larger is better), skewness and kurtosis of the standardized
residuals, Ljung-Box statistics for 20 lags, and the three loss functions of Section 2.
"""
function diagnostics(ε, h, LL, k)
	z = ε ./ sqrt.(h)
	ljungbox(x) = (c = x .- mean(x); T * (T + 2) * sum((sum(c[j+1:end] .* c[1:end-j]) / sum(abs2, c))^2 / (T - j) for j in 1:20))
	(AIC = 2LL - 2k, SIC = 2LL - k * log(T), b3 = mean(z .^ 3), b4 = mean(z .^ 4), Q = ljungbox(z), Q2 = ljungbox(z .^ 2),
		MSE = mean((ε .^ 2 .- h) .^ 2), HMSE = mean((ε .^ 2 ./ h .- 1) .^ 2), LL = mean(log.(ε .^ 2 ./ h) .^ 2))
end

"""
	qmle(free, start)

Quasi-maximum likelihood estimates of the parameters numbered in `free` (the others are zero), from `start`, with robust standard errors.
"""
function qmle(free, start)
	expand(p) = (θ = zeros(eltype(p), 7); θ[free] = p; θ)
	nll(p) = -sum(loglik(expand(p)))
	p = Optim.minimizer(optimize(nll, start, BFGS(), Optim.Options(g_tol=1e-10); autodiff=:forward))
	H = ForwardDiff.hessian(nll, p)
	G = ForwardDiff.jacobian(q -> loglik(expand(q)), p)  # the score of each day
	se = sqrt.(diag(H \ (transpose(G) * G) / H))  # robust (sandwich) standard errors
	ε, h = variances(expand(p))
	(θ = expand(p), se = expand(se), h = h, logL = -nll(p), diagnostics(ε, h, -nll(p), length(p))...)
end

"""
	table2(title, m, published)

Replicate a column of Table 2: `published` holds each estimate with its standard error, and the statistics, as printed.
"""
function table2(title, m, published)
	rows = []
	for (i, name) in enumerate(parameters)
		haskey(published, Symbol(name)) || continue
		b, se = published[Symbol(name)]
		push!(rows, (name, m.θ[i], b), ("se " * name, m.se[i], se))
	end
	for name in statistics
		push!(rows, (string(name), m[name], published[name]))
	end
	replicate(title, rows)
end

## 3. A constant variance: is it larger after no trading?

The first three columns of Table 2 keep the variance constant, apart from the non-trading-day effect: ω₁ is the extra variance after a weekend or holiday, μ₁ the change in the mean. There is no mean effect, but ω₁ is highly significant: the variance is about 50% larger after no trading (p. 145). The large Q²(20), the Ljung and Box (1978) statistic of the squared standardized residuals, shows the volatility clustering that a constant variance leaves out.

In [ ]:
m1 = qmle([1, 3], [0.0, 0.2]);  # mu, omega
m2 = qmle([1, 2, 3, 4], [0.0, 0.0, 0.2, 0.1]);  # mu, mu1, omega, omega1
m3 = qmle([1, 3, 4], [0.0, 0.2, 0.1]);  # mu, omega, omega1
table2("Constant variance (Bollerslev and Ghysels 1996, p. 146, Table 2, column 1)", m1, (mu = ("-.016", ".011"), omega = (".221", ".012"),
	AIC = "-2,626.2", SIC = "-2,652.5", b3 = "-.25", b4 = "6.63", Q = "27.8", Q2 = "507.6", MSE = ".275", HMSE = "5.63", LL = "9.67"))
table2("Non-trading days in mean and variance (Table 2, column 2)", m2, (mu = ("-.013", ".011"), mu1 = ("-.017", ".028"), omega = (".200", ".011"), omega1 = (".092", ".036"),
	AIC = "-2,602.7", SIC = "-2,625.1", b3 = "-.25", b4 = "6.18", Q = "27.2", Q2 = "593.0", MSE = ".273", HMSE = "5.18", LL = "9.87"))
table2("Non-trading days in the variance (Table 2, column 3)", m3, (mu = ("-.015", ".010"), omega = (".200", ".011"), omega1 = (".092", ".036"),
	AIC = "-2,601.1", SIC = "-2,617.9", b3 = "-.25", b4 = "6.18", Q = "26.9", Q2 = "599.4", MSE = ".273", HMSE = "5.18", LL = "9.52"))
@printf("\nvariance after no trading: %.0f%% larger (omega1 / omega)\n", 100 * m3.θ[4] / m3.θ[3])

## 4. GARCH(1,1) with ARCHModels

The fourth column lets the variance follow its own past: GARCH(1,1), fitted here with ARCHModels. ARCHModels writes the intercept of the variance as ω(1 − α − β), so its ω is converted to the level the variance returns to, the paper's ω, with a delta-method standard error. The statistics come from the package's residuals and volatilities. Q²(20) falls from about 600 to 17.5: GARCH captures the clustering.

In [ ]:
am = fit(GARCH{1, 1}, y; meanspec=Intercept)  # robust standard errors by default
w, b1, a1, mu = coef(am);  # intercept, beta, alpha, mean
level = w / (1 - a1 - b1);  # the paper's omega
grad = [1, level, level, 0] ./ (1 - a1 - b1);  # its derivative with respect to (w, b1, a1, mu)
se_am = sqrt.(diag(vcov(am)));
m4 = (logL = loglikelihood(am), diagnostics(residuals(am; standardized=false), volatilities(am) .^ 2, loglikelihood(am), 4)...);
replicate("GARCH(1,1), ARCHModels (Table 2, column 4)", [
	("mu", mu, "-.006"), ("se mu", se_am[4], ".009"), ("omega", level, ".264"), ("se omega", sqrt(transpose(grad) * vcov(am) * grad), ".075"),
	("alpha", a1, ".153"), ("se alpha", se_am[3], ".054"), ("beta", b1, ".806"), ("se beta", se_am[2], ".073"),
	[(string(name), m4[name], p) for (name, p) in zip(statistics, ["-2,221.2", "-2,243.5", "-.40", "6.56", "19.3", "17.5", ".253", "5.52", "8.79"])]...])

## 5. GARCH(1,1) with the non-trading-day dummy

The fifth column adds ω₁ to GARCH(1,1), entered as an impulse that lasts one day (p. 146, following Baillie and Bollerslev 1989). No Julia package has a dummy in the variance equation, so this is `qmle` with ω₁ set free; the same code without ω₁ gives ARCHModels' estimates above. The effect remains very significant, and both AIC and SIC prefer it.

In [ ]:
m5 = qmle([1, 3, 4, 5, 7], [0.0, 0.25, 0.05, 0.15, 0.8]);  # mu, omega, omega1, alpha, beta
table2("GARCH(1,1) with the dummy (Table 2, column 5)", m5, (mu = ("-.009", ".008"), omega = (".259", ".086"), omega1 = (".054", ".022"), alpha = (".137", ".035"), beta = (".832", ".044"),
	AIC = "-2,191.0", SIC = "-2,219.0", b3 = "-.44", b4 = "5.89", Q = "18.8", Q2 = "21.7", MSE = ".252", HMSE = "4.88", LL = "8.21"))

## 6. Periodic GARCH: a new method

The last column is the paper's new model: P-GARCH(1,1), in which α, how much a shock feeds into tomorrow's variance, takes one value on ordinary days and another, α + α₁, after no trading. It is `qmle` with α₁ free as well: the method is a few lines once the likelihood is written down, and the published estimates are its test.

In [ ]:
m6 = qmle([1, 3, 4, 5, 6, 7], [0.0, 0.3, 0.05, 0.15, -0.1, 0.8]);  # mu, omega, omega1, alpha, alpha1, beta
table2("P-GARCH(1,1) (Table 2, column 6)", m6, (mu = ("-.006", ".008"), omega = (".341", ".169"), omega1 = (".043", ".026"), alpha = (".178", ".043"),
	alpha1 = ("-.111", ".044"), beta = (".822", ".042"),
	AIC = "-2,179.3", SIC = "-2,212.9", b3 = "-.46", b4 = "5.87", Q = "19.1", Q2 = "23.9", MSE = ".257", HMSE = "4.67", LL = "8.66"))
replicate("Persistence of a shock (p. 146)", [("alpha + beta", m6.θ[5] + m6.θ[7], "1.000"), ("alpha + alpha1 + beta", m6.θ[5] + m6.θ[6] + m6.θ[7], ".889")])

days = 1400:1600;  # a stretch of 1989-1990
plot(days, sqrt.(m5.h[days]), label="GARCH(1,1) with the dummy", xlabel="trading day", ylabel="conditional volatility (%)")
plot!(days, sqrt.(m6.h[days]), label="P-GARCH(1,1)")

## 7. The branch: GARCH or periodic GARCH?

Shocks on ordinary days persist (α + β is 1.000), while those after no trading die out faster (.889): news that arrives while the market is closed says less about tomorrow (p. 146). Both information criteria, larger is better, and the robust t-statistic of α₁ favour the periodic model; if they do, `periodic` is true and the P-GARCH estimates are the ones to keep, otherwise GARCH(1,1) with the dummy.

In [ ]:
@printf("%-34s%12s%12s%12s\n", "", "log lik.", "AIC", "SIC")
for (label, m) in [("Constant variance", m1), ("Non-trading days, mean and var.", m2), ("Non-trading days, variance", m3),
		("GARCH(1,1)", m4), ("GARCH(1,1) with the dummy", m5), ("P-GARCH(1,1)", m6)]
	@printf("%-34s%12.1f%12.1f%12.1f\n", label, m.logL, m.AIC, m.SIC)
end
t_alpha1 = m6.θ[6] / m6.se[6];
@printf("\nrobust t-statistic of alpha1: %.2f\n", t_alpha1)
periodic = abs(t_alpha1) > 1.96 && m6.AIC > m5.AIC && m6.SIC > m5.SIC  # the rule: significant at 5% and preferred by both criteria

## 8. What differs, and why

Most values agree to the published precision; the rest are listed here, explained where the cause was found.

- **Column 1, b4 and HMSE (6.62 and 5.62 against 6.63 and 5.63).** Both depend on μ and ω beyond the three printed decimals: at the published μ = −.016 and ω = .221 they give the published values (the cell below).
- **Column 1, SIC (−2,637.4 against −2,652.5).** A misprint: every other column, and column 1's own AIC, fit AIC = 2 log L − 2k and SIC = 2 log L − k ln T, and no variant gives −2,652.5.
- **Columns 1 and 2, the LL loss (9.71 and 9.67 against 9.67 and 9.87).** The mean of ln(ε²/σ²)² is dominated by the days whose return is closest to the mean, so it moves with the fourth decimal of μ (at the published μ, column 1 gives 9.83). Column 2's 9.87 is not explained.
- **Estimates on a rounding boundary.** Column 5's α is .1375 and column 6's α is .1785 (published .137 and .178), and α + β is 1.0006 (published 1.000): they round the other way.
- **One unit in the last digit.** AIC, SIC, Q²(20), b3 or b4 in columns 2, 3 and 5, and the standard errors of ω₁ in column 5 and of α in column 6. The statistics are computed from the estimates, so they move with digits beyond the three printed; these differences are of that size.
- **Standard errors in the third decimal (e.g. column 6, ω: .171 against .169).** These are the parameters along which the likelihood is flattest, where the result depends on how the Hessian is computed: numerically in 1996, exactly here with ForwardDiff. Likely, not proven.
- **Columns 4 and 6, b3 and b4 (P-GARCH: −.43 and 5.63 against −.46 and 5.87).** Not explained. One Monday, day 1,525 (return −2.14), carries most of the sum of z⁴; the published b4 would need its standardized residual near −6.0 rather than −5.5. How the recursion starts, dropping the first day, and switching α₁ by the previous day's dummy instead were all tried: none gives the published values, and the last one changes the estimates.

Ruled out on the way: the robust standard errors of Bollerslev and Wooldridge (1992) (they give about half the published ones for α and β), so the paper's are the sandwich used here.

In [ ]:
ε, h = variances([-0.016, 0, 0.221, 0, 0, 0, 0]);  # column 1 at its published estimates
d1 = diagnostics(ε, h, sum(loglik([-0.016, 0, 0.221, 0, 0, 0, 0])), 2);
replicate("Column 1 at the published estimates (Table 2)", [("b4", d1.b4, "6.63"), ("HMSE", d1.HMSE, "5.63"), ("LL", d1.LL, "9.67")])

**Now in Compose › New Methods.** P-GARCH is not in any Julia package. The New Methods notebook **Periodic GARCH: volatility with a calendar dummy** writes it out as documented functions, tested against Table 2 of this paper and, for plain GARCH, against ARCHModels.jl. Open it from **Compose › New Methods**. The paper's second example (Table 3), intraday Deutsche mark / US dollar returns, uses data that is not public.

## References

- Baillie, R. T. and Bollerslev, T. (1989). The message in daily exchange rates: a conditional-variance tale. *Journal of Business & Economic Statistics* 7(3), 297-305.
- Bollerslev, T. and Ghysels, E. (1996). Periodic autoregressive conditional heteroscedasticity. *Journal of Business & Economic Statistics* 14(2), 139-151. doi:10.1080/07350015.1996.10524640
- Bollerslev, T. and Wooldridge, J. M. (1992). Quasi-maximum likelihood estimation and inference in dynamic models with time-varying covariances. *Econometric Reviews* 11(2), 143-172.
- Ljung, G. M. and Box, G. E. P. (1978). On a measure of lack of fit in time series models. *Biometrika* 65(2), 297-303.
- Data: the returns as distributed with ARCHModels.jl, and the non-trading-day dummy as distributed with the R package rugarch (`dmbp`).